In [12]:
# 1. 필요한 라이브러리 설치
%pip install requests beautifulsoup4 pandas surya-ocr google-genai cloudscraper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [cloudscraper]
Note: you may need to restart the kernel to use updated packages.


# 인디스워크 채용 공고 크롤러 (최종 완성판)
- 공고 목록 자동 수집 (페이지 넘김 지원)
- **데이터 분리 저장**:
  1. `content_text`: 순수 본문 텍스트 (지원 버튼 문구 제거됨)
  2. `content_images`: 본문 내 이미지 URL들
  3. `apply_url`: '지원하러 가기' 버튼의 실제 링크
- 결과 CSV 파일로 저장

In [24]:
import cloudscraper
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os

# -----------------------------------------------------
# 설정
# -----------------------------------------------------
TARGET_PAGES = 4

scraper = cloudscraper.create_scraper()

def get_headers(referer="https://inthiswork.com/"):
    """
    맥(Mac) 크롬 브라우저와 최대한 비슷하게 헤더를 구성합니다.
    """
    return {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': referer,
        'DNT': '1',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'same-origin',
        'Sec-Fetch-User': '?1'
    }

def get_job_list(pages=3):
    print("=== 공고 목록 수집 시작 ===")
    all_jobs = []
    
    # 세션을 사용하면 쿠키가 유지되어 더 사람처럼 인식될 가능성이 높습니다.
    session = requests.Session()
    for page in range(1, pages + 1):
        url = "https://inthiswork.com/it" if page == 1 else f"https://inthiswork.com/it?paged1={page}"
        referer = "https://inthiswork.com/" if page == 1 else "https://inthiswork.com/it"
            
        time.sleep(random.uniform(2, 4))
        print(f"[{page}페이지] 목록 수집 중... ({url})")
        
        try:
            resp = scraper.get(url, headers=get_headers(referer), timeout=20)
            
            # 차단 여부 확인용 로그
            if resp.status_code != 200:
                print(f"  → 에러: 접속 실패 (상태 코드 {resp.status_code})")
                continue
                
            soup = BeautifulSoup(resp.text, 'html.parser')
            
            # 핵심 변경: 컨테이너에 국한되지 않고 페이지 전체에서 공고 항목 직접 탐색
            entries = soup.select('.dpt-entry')
            
            if not entries:
                # 만약 항목을 못 찾았다면, 다른 클래스명으로 다시 시도
                entries = soup.select('.dpt-entry-wrapper')
            
            print(f"  → {len(entries)}개 공고 발견")
            for entry in entries:
                # 링크와 제목이 포함된 태그 찾기
                link_obj = entry.select_one('a.dpt-title-link') or entry.select_one('.dpt-title a')
                
                if link_obj and link_obj.get('href'):
                    full_url = link_obj.get('href')
                    full_text = link_obj.get_text(strip=True)
                    
                    if '/archives/' not in full_url: 
                        continue
                    # 제목 구분자 처리 (공백 및 특수문자 대응)
                    if '｜' in full_text:
                        parts = full_text.split('｜', 1)
                    elif '|' in full_text:
                        parts = full_text.split('|', 1)
                    else:
                        parts = ["-", full_text]
                    
                    company = parts[0].strip() if len(parts) > 1 else "-"
                    title = parts[1].strip() if len(parts) > 1 else parts[0].strip()

                    # 회사명이 없거나(-) 혹은 "IN THIS WORK"를 포함하는 경우(인터뷰/콘텐츠) 제외
                    if company == "-" or "IN THIS WORK" in company.upper():
                        print(f"  → 제외: {company} | {title}") # 제외되는 공고 로그 출력 (선택 사항)
                        continue
                    
                    # 중복 체크
                    if not any(j['url'] == full_url for j in all_jobs):
                        all_jobs.append({
                            'company': company,
                            'title': title,
                            'url': full_url
                        })
                        
        except Exception as e:
            print(f"  → 에러 발생: {e}")
    return all_jobs

def get_job_detail(url):
    """
    상세 정보 추출 (텍스트, 이미지, 지원링크 분리)
    """
    time.sleep(random.uniform(4, 8))
    try:
        resp = scraper.get(url, headers=get_headers("https://inthiswork.com/it"), timeout=20)

        if resp.status_code != 200:
            print(f"  → 접속 거부됨 ({resp.status_code}): {url}")
            return "", "", ""


        soup = BeautifulSoup(resp.text, 'html.parser')
        
        content_area = (
            soup.select_one('.fusion-content-tb') or 
            soup.select_one('.fusion-content-tb-1') or 
            soup.select_one('.fusion-content-tb-2') or 
            soup.select_one('.post-content') or 
            soup.select_one('.entry-content')
        )
        
        # 영역 찾기 실패 시 디버깅을 위해 HTML 길이를 출력해봅니다.
        if not content_area:
            print(f"  → 내용 영역 미발견 (HTML 길이: {len(resp.text)}): {url}")
            # 만약 보안 페이지가 아니라면, 최소한의 텍스트라도 가져오기 위해 body 전체를 설정
            if len(resp.text) > 2000: # 정상적인 페이지로 보일 때
                content_area = soup.find('body')
            else:
                return "", "", ""
        
        # 1. 지원하기 버튼 링크 추출 (.maxbutton 또는 텍스트로 찾기)
        apply_link_obj = content_area.select_one('a.maxbutton')
        if not apply_link_obj:
            # 클래스가 없을 경우 텍스트로 찾기
            for a in content_area.select('a'):
                if "지원하러" in a.get_text() or "지원하기" in a.get_text():
                    apply_link_obj = a
                    break
        
        apply_url = ""
        if apply_link_obj:
            apply_url = apply_link_obj.get('href', '')
            # 본문 텍스트에서 '지원하러 가기' 문구를 없애기 위해 태그 삭제
            apply_link_obj.decompose()

        # 2. 이미지 추출 (decompose 전에 해야 함, 하지만 버튼은 이미지와 상관없으므로 순서 무관)
        images = content_area.select('img')
        image_links = [img.get('src') for img in images if img.get('src')]
        images_str = ", ".join(image_links)

        # 3. 텍스트 추출 (버튼 태그가 삭제된 상태에서 추출)
        text_content = content_area.get_text(separator='\n', strip=True)
        
        return text_content[:5000], images_str, apply_url

    except Exception as e:
        print(f"  → 상세 에러({url}): {e}")
        return "", "", ""

# --- 실행 ---
job_list = get_job_list(pages=TARGET_PAGES)

full_data = []
if job_list:
    print("\n=== 상세 정보 수집 시작 ===")
    for idx, job in enumerate(job_list):
        print(f"[{idx+1}/{len(job_list)}] {job['title'][:15]}...")
        
        # 3가지 정보 받아오기
        text, imgs, apply_url = get_job_detail(job['url'])
        
        job['content_text'] = text
        job['content_images'] = imgs
        job['apply_url'] = apply_url
        
        full_data.append(job)

    # 저장
    df = pd.DataFrame(full_data)
    save_file = "recruitment_results_full.csv"

    if os.path.exists(save_file):
        # 파일이 있으면: 헤더 없이(header=False), 추가 모드(mode='a')로 저장
        df.to_csv(save_file, mode='a', header=False, index=False, encoding='utf-8-sig')
        print(f"\n[추가 완료] 기존 파일 뒤에 {len(df)}건이 추가되었습니다.")
    else:
        # 파일이 없으면: 헤더 포함(header=True), 쓰기 모드(mode='w')로 신규 생성
        df.to_csv(save_file, mode='w', header=True, index=False, encoding='utf-8-sig')
        print(f"\n[저장 완료] 새 파일에 {len(df)}건이 저장되었습니다.")
    
    # 결과 확인 (apply_url 컬럼 추가됨)
    pd.set_option('display.max_columns', None)
    
    # 현재 수집된 데이터만 확인
    display(df[['company', 'title', 'apply_url']].head())

=== 공고 목록 수집 시작 ===
[1페이지] 목록 수집 중... (https://inthiswork.com/it)
  → 44개 공고 발견
  → 제외: - | 2026 상반기 S-개발자 4기 모집
  → 제외: IN THIS WORK 인터뷰 | 한국딥러닝 채용
  → 제외: - | 스타트업 vs 대기업, 경험자들이 알려주는 스타트업과 대기업의 차이!ㅣ코딥토크쇼
  → 제외: - | 신입 개발자 채용 공고 분석해 드립니다
  → 제외: - | 신입 개발자 채용 트렌드 총정리.zip
  → 제외: - | 개발자 이력서, 제발 이렇게 쓰지 마세요!
[2페이지] 목록 수집 중... (https://inthiswork.com/it?paged1=2)
  → 44개 공고 발견
  → 제외: - | 2026년 상반기 현대케피코 신입채용(채용연계형 인턴)
  → 제외: - | 2026년 상반기 DY그룹 신입/경력 채용
[3페이지] 목록 수집 중... (https://inthiswork.com/it?paged1=3)
  → 44개 공고 발견
  → 제외: - | 환인제약(주) 상시 인재pool 등록
  → 제외: - | 소상공인 힘보탬 박람회 대학생 아이디어 경진대회
[4페이지] 목록 수집 중... (https://inthiswork.com/it?paged1=4)
  → 44개 공고 발견
  → 제외: - | 2026년 상반기 코오롱베니트 채용연계형 인턴 모집

=== 상세 정보 수집 시작 ===
[1/130] Security Resear...
[2/130] 2026년 상반기 신입/경력...
[3/130] [신입/경력] 백엔드 엔지니...
[4/130] AI Software Eng...
[5/130] Robotics S/W En...
[6/130] 3D Computer Vis...
[7/130] [연구소] 네트워크보안솔루션...
[8/130] [연구소] AI솔루션 개발자...
[9/130] Product Enginee...
[10/130] 대외 SI/SM 사업 프로젝...

,company,title,apply_url
0,토스뱅크,Security Research Specialist,https://toss.im/career/job-detail?job_id=75698...
1,세아제강,2026년 상반기 신입/경력 사원채용,https://seahsteel.recruiter.co.kr/career/job
2,포스타입,[신입/경력] 백엔드 엔지니어,https://about.postype.com/ko/o/199867
3,망고부스트,"AI Software Engineer(Seoul, South Korea)",https://mangoboost.career.greetinghr.com/en/o/...
4,리얼월드(RLWRLD),Robotics S/W Engineer,https://realworld.career.greetinghr.com/ko/o/1...


## 이미지 형식 공고를 OCR로 텍스트로 변환

In [14]:
# --- [2단계] 이미지 OCR 처리 (텍스트 없는 공고 변환) ---
# 사전 설치 필요: %pip install surya-ocr

import requests
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO

# == Surya OCR 최신 버전 API 임포트 ==
from surya.detection import DetectionPredictor
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.common.surya.schema import TaskNames

# 모델 로드 (전역에서 한 번만 수행)
# 처음 실행 시 모델 가중치를 다운로드하므로 시간이 조금 걸릴 수 있습니다.
print("Surya OCR 모델 로딩 중...")
foundation_predictor = FoundationPredictor()
det_predictor = DetectionPredictor()
rec_predictor = RecognitionPredictor(foundation_predictor)
print("모델 로딩 완료.")

def read_text_from_image_url(url):
    """
    URL 이미지를 읽어서 Surya OCR로 텍스트를 반환하는 함수
    """
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200: return ""
        
        image_bytes = BytesIO(response.content)
        image = Image.open(image_bytes).convert("RGB")
        
        # 최신 API: Predictor 객체를 직접 호출합니다.
        # 인식할 언어는 자동으로 감지하거나 settings에서 설정됩니다.
        predictions = rec_predictor(
            [image], 
            [TaskNames.ocr_with_boxes], 
            det_predictor=det_predictor
        )
        
        # 결과 텍스트 추출
        if predictions and predictions[0].text_lines:
            full_text = "\n".join([line.text for line in predictions[0].text_lines])
            return full_text
        
        return ""
    except Exception as e:
        print(f"OCR 에러 ({url}): {e}")
        return ""

/data/ephemeral/git/pro-nlp-generationfornlp-nlp-01/.venv/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Surya OCR 모델 로딩 중...


모델 로딩 완료.


In [15]:
# 1. CSV 불러오기
df = pd.read_csv("recruitment_results_full.csv")

# 2. '텍스트는 없고(부족하고) 이미지는 있는' 행 찾기
target_mask = (df['content_text'].fillna('').str.len() < 50) & (df['content_images'].notna()) & (df['content_images'] != "")
targets = df[target_mask]
print(f"OCR 변환 대상: 총 {len(targets)}개 공고")

# 3. 순회하며 OCR 수행
for idx, row in targets.iterrows():
    print(f"[{idx}] '{row['title']}' 이미지 분석 중...")
    
    img_urls = str(row['content_images']).split(',')
    ocr_text_all = []
    
    for img_url in img_urls:
        img_url = img_url.strip()
        if not img_url: continue
        
        text = read_text_from_image_url(img_url)
        if text: ocr_text_all.append(text)
            
    if ocr_text_all:
        full_ocr_text = "(이미지 자동 추출 텍스트)\n" + "\n---\n".join(ocr_text_all)
        df.at[idx, 'content_text'] = full_ocr_text
        print("  -> 변환 완료")

# 4. 결과 재저장
df.to_csv("recruitment_results_full_ocr.csv", index=False, encoding='utf-8-sig')
print("\n[완료] 'recruitment_results_full_ocr.csv'가 저장되었습니다.")

OCR 변환 대상: 총 10개 공고
[1] '2026년 상반기 신입/경력 사원채용' 이미지 분석 중...


/data/ephemeral/git/pro-nlp-generationfornlp-nlp-01/.venv/lib/python3.10/site-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Recognizing Text: 100%|██████████| 321/321 [00:17<00:00, 18.07it/s] 


  -> 변환 완료
[9] '대외 SI/SM 사업 프로젝트 관리자(PM) 및 개발자 모집' 이미지 분석 중...


Recognizing Text: 100%|██████████| 62/62 [00:02<00:00, 25.18it/s]


  -> 변환 완료
[16] '정보보호 (모의해킹/취약점진단, 침해사고대응)' 이미지 분석 중...


...

  -> 변환 완료
[25] 'SAP RAP(Restful ABAP) 개발자 모집' 이미지 분석 중...


Recognizing Text: 100%|██████████| 53/53 [00:01<00:00, 28.04it/s]


  -> 변환 완료
[26] '2026년 신입사원 채용' 이미지 분석 중...


Recognizing Text: 100%|██████████| 425/425 [00:10<00:00, 40.33it/s] 


  -> 변환 완료

[완료] 'recruitment_results_full_ocr.csv'가 저장되었습니다.


# 크롤링 후 Json 형식으로 생성하기

## 크롤링 정보를 저장해 놓은 csv 파일에서 데이터 읽어오기

In [20]:
import pandas as pd

# 1. 수집된 CSV 파일 불러오기
# 전처리가 완료된 최종 파일을 불러옵니다.
csv_path = "recruitment_results_full_ocr.csv"

try:
    df = pd.read_csv(csv_path, encoding='utf-8-sig')
    print(f"✅ 파일을 성공적으로 불러왔습니다. (총 {len(df)}개 공고)")
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다: {csv_path}")

# 2. 데이터 최종 점검 (결측치 처리)
# 분석 시 에러 방지를 위해 결측치를 빈 문자열이나 기본값으로 채웁니다.
df['content_text'] = df['content_text'].fillna('')
# df['content_images'] = df['content_images'].fillna('')
df['company'] = df['company'].fillna('정보 없음')
df['title'] = df['title'].fillna('제목 없음')

# 3. 대상 지정 (필터링 없이 전체 데이터 사용)
# 이제 특정 조건(이미지 유무 등)에 관계없이 모든 데이터를 분석 대상으로 설정합니다.
df_targets = df.copy() 

print(f"📊 분석 대상 데이터: 총 {len(df_targets)}건 확보 완료")

# 4. 전체 데이터 구조 확인 (상위 5건)
print("\n--- 전체 데이터 샘플 (최상위 5건) ---")
# 필요한 컬럼들만 선택해서 확인
display(df_targets[['company', 'title', 'content_text']].head())

# 이후 단계에서 df_targets를 사용하여 LLM 분석이나 결과 저장을 진행하시면 됩니다.

✅ 파일을 성공적으로 불러왔습니다. (총 40개 공고)
📊 분석 대상 데이터: 총 40건 확보 완료

--- 전체 데이터 샘플 (최상위 5건) ---


,company,title,content_text,content_images
0,토스뱅크,Security Research Specialist,토스뱅크 Affiliation\n계약직\n합류하게 될 팀에 대해 알려드려요\nSec...,
1,세아제강,2026년 상반기 신입/경력 사원채용,(이미지 자동 추출 텍스트)\nSěAH 세아제강\n2026\n세아제강\n신입/경력사...,https://i0.wp.com/inthiswork.com/wp-content/up...
2,포스타입,[신입/경력] 백엔드 엔지니어,취향의 가치를 ​만드는 ​창작 ​커뮤니티\n포스타입에서는 다양한 ​취향을 만족하는 ...,
3,망고부스트,"AI Software Engineer(Seoul, South Korea)",Position Overview\nWe ​are ​looking ​for a ​hi...,
4,리얼월드(RLWRLD),Robotics S/W Engineer,리얼월드\n(RLWRLD)는 로봇이 ​인간처럼 ​세상을 ​인식하고 사고하며 ​행동할...,


## Gemini 설정 및 프롬프트 준비

In [ ]:
from google import genai # import google.genai as genai 대신 이 방식을 권장합니다.
from google.genai import types
import PIL.Image
import requests
from io import BytesIO
import time
import json
import pandas as pd

# 1. 클라이언트(Client) 객체를 생성하며 API 키를 전달합니다.
GOOGLE_API_KEY = ""
client = genai.Client(api_key=GOOGLE_API_KEY)

# 2. 모델명 설정
MODEL_ID = 'gemini-2.5-flash' # 또는 'gemini-2.5-flash' (사용 가능 환경인 경우)
SYSTEM_PROMPT = """채용 공고 전문가로서 다음 지침에 따라 공고 데이터를 분석하여 JSON 리스트로 변환하세요.
1. 한 공고 내에 여러 직무(백엔드, 프론트엔드 등)가 있다면 각각 독립된 JSON 객체로 분리하여 리스트로 만드세요.
2. 표(Table) 형식의 데이터는 행과 열의 관계를 정확히 파악하여 각 직무에 맞게 할당하세요.
3. link에는 실제 지원 페이지로 연결되는 apply_url 정보를 담아주세요.
4. 출력 형식은 반드시 아래 키를 포함하는 JSON 리스트여야 합니다:
   keys: [title, company, link, deadline, location, experience, education, employment_type, salary, job_sector, key_responsibilities, required_qualifications, preferred_qualifications]
"""

## 방식 A: 로컬 OCR (Surya) -> 텍스트 -> Gemini

로컬에서 surya-ocr로 텍스트를 모두 뽑아낸 뒤, 텍스트만 Gemini에게 전달합니다.



In [28]:
def approach_a_ocr_then_gemini(df_row):
    # 1. 기존 텍스트 가져오기
    existing_text = str(df_row['content_text']) if pd.notna(df_row['content_text']) else ""
    
    # 2. 이미지 OCR 수행 (img_urls 추출)
    img_urls = str(df_row['content_images']).split(',') if pd.notna(df_row['content_images']) else []
    ocr_text_all = ""
    
    for url in img_urls:
        url = url.strip()
        if not url: continue
        # 이전에 정의한 Surya OCR 함수 호출
        text = read_text_from_image_url(url) 
        ocr_text_all += f"\n[이미지 추출 텍스트]:\n{text}\n"

    # 3. 통합 텍스트 구성
    combined_input = f"""
    회사명: {df_row['company']}
    공고 제목: {df_row['title']}
    원본 링크: {df_row['url']}
    지원 링크: {df_row['apply_url']}
    
    [본문 텍스트]:
    {existing_text}
    
    {ocr_text_all}
    """

    # 4. Gemini 호출 (신규 SDK 문법)
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=f"다음 데이터를 분석해:\n{combined_input}",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json"
        )
    )
    return response.text

## 방식 B : Gemini 멀티모달 (이미지 직접 전달)

Gemini는 이미지 URL을 직접 인식하지 못하므로, 이미지를 다운로드하여 바이트 데이터로 직접 전달

In [29]:
def approach_b_url_direct_gemini(df_row):
    img_urls = str(df_row['content_images']).split(',') if pd.notna(df_row['content_images']) else []
    img_urls_str = "\n".join([f"- {url.strip()}" for url in img_urls if url.strip()])
    existing_text = str(df_row['content_text']) if pd.notna(df_row['content_text']) else ""

    user_prompt = f"""
    다음 채용 공고의 텍스트와 이미지 URL들을 분석하여 정보를 추출하세요.
    
    기본 정보:
    - 회사명: {df_row['company']}
    - 제목: {df_row['title']}
    - 공고 URL: {df_row['url']}
    - 지원 URL: {df_row['apply_url']}
    
    본문 텍스트:
    {existing_text}
    
    분석할 이미지 URL 목록:
    {img_urls_str}
    """

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type="application/json"
        )
    )
    return response.text

## 비교 실험 실행 코드

In [ ]:
results_comparison = []

def run_gemini_experiment(df_row):
    print(f"\n>>> 분석 중: [{df_row['company']}] {df_row['title']}")
    
    # --- 방식 A: OCR 기반 ---
    try:
        start_a = time.time()
        res_a = approach_a_ocr_then_gemini(df_row)
        time_a = time.time() - start_a
    except Exception as e:
        res_a, time_a = f"Error: {e}", 0

    # --- 방식 B: URL 직접 전달 기반 ---
    try:
        start_b = time.time()
        res_b = approach_b_url_direct_gemini(df_row)
        time_b = time.time() - start_b
    except Exception as e:
        res_b, time_b = f"Error: {e}", 0
    
    results_comparison.append({
        'company': df_row['company'],
        'title': df_row['title'],
        'time_ocr_gemini': time_a,
        'time_url_context': time_b,
        'result_a_len': len(res_a),
        'result_b_len': len(res_b),
        'raw_a': res_a,
        'raw_b': res_b
    })

# 1. 파일 읽기
df = pd.read_csv("recruitment_results_full.csv")

# 2. 테스트용 샘플 추출 (이미지가 있는 데이터 우선)
samples = df[df['content_images'].notna() & (df['content_images'] != "")].head(3)

# 3. 실행
for _, row in samples.iterrows():
    run_gemini_experiment(row)

# 4. 결과 출력
comparison_df = pd.DataFrame(results_comparison)
pd.set_option('display.max_colwidth', 50)
display(comparison_df[['company', 'title', 'time_ocr_gemini', 'time_url_context']])

output_dir = "experiment_results"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
for i, res in enumerate(results_comparison):
    # 회사명과 인덱스로 파일명 생성 (특수문자 제거)
    clean_company = "".join(x for x in res['company'] if x.isalnum())
    
    # 방식 A 결과 저장
    with open(f"{output_dir}/{i}_{clean_company}_approach_A.json", "w", encoding="utf-8") as f:
        f.write(res['raw_a'])
    
    # 방식 B 결과 저장
    with open(f"{output_dir}/{i}_{clean_company}_approach_B.json", "w", encoding="utf-8") as f:
        f.write(res['raw_b'])
print(f"\n📂 상세 JSON 결과가 '{output_dir}' 폴더에 저장되었습니다.")
print("각 파일을 열어서 비교해 보세요!")
# 6. 주피터 노트북에서 바로 내용 일부 확인하기 (첫 번째 샘플)
if results_comparison:
    print("\n--- [직접 확인] 첫 번째 샘플의 방식 B 결과 ---")
    print(results_comparison[0]['raw_b'])

## OCR 완료된 내용들을 json 형식으로 바꾸는 요청

In [ ]:
def analyze_with_gemini(df_row):
    # 1. 가공된 텍스트 가져오기 (이미 OCR 등이 완료된 상태)
    full_text = str(df_row['content_text']) if pd.notna(df_row['content_text']) else ""
    
    # 2. 통합 프롬프트 구성
    combined_input = f"""
    회사명: {df_row['company']}
    공고 제목: {df_row['title']}
    원본 링크: {df_row.get('url', 'N/A')}
    지원 링크: {df_row.get('apply_url', 'N/A')}
    
    [본문 및 OCR 텍스트]:
    {full_text}
    """

    try:
        # 3. Gemini 호출
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=[f"다음 채용 공고를 상세히 분석해서 JSON으로 응답해줘:\n{combined_input}"],
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                response_mime_type="application/json"
            )
        )
        
        # JSON 문자열을 파이썬 딕셔너리로 변환
        import json
        result = json.loads(response.text)
        
        # 결과를 담을 리스트 정형화
        # 1. 단일 딕셔너리면 리스트로 감쌈
        # 2. 이미 리스트라면 그대로 사용
        items_to_process = result if isinstance(result, list) else [result]
        
        # 각 결과물에 공통 정보(회사명, 제목 등) 추가
        for item in items_to_process:
            if isinstance(item, dict): # 딕셔너리인 경우만 키 추가
                item['source_company'] = df_row['company']
                item['source_title'] = df_row['title']
                item['source_url'] = df_row.get('url', '')
        
        return items_to_process # 항상 리스트 형태 반환

    except Exception as e:
        print(f"❌ 분석 실패 ({df_row['title']}): {e}")
        return {"error": str(e), "title": df_row['title']}

## 저장하기

In [23]:
import json
from tqdm import tqdm

final_all_results = [] # 모든 공고에서 추출된 모든 JSON 객체가 여기 담깁니다.

print(f"🚀 총 {len(df_targets)}건 분석 시작...")

for index, row in tqdm(df_targets.iterrows(), total=len(df_targets)):
    # 결과가 리스트로 옵니다 (예: [{}, {}] 또는 [{}])
    batch_results = analyze_with_gemini(row)
    
    # extend를 사용하여 합칩니다.
    final_all_results.extend(batch_results)

# 3. 저장
output_filename = "final_recruitment_all_items.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(final_all_results, f, ensure_ascii=False, indent=4)

print(f"✅ 총 {len(final_all_results)}개의 데이터가 '{output_filename}'에 저장되었습니다.")

🚀 총 40건 분석 시작...


 22%|██▎       | 9/40 [01:55<03:41,  7.16s/it]

❌ 분석 실패 (Product Engineer (Full-Stack)): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 34.458896721s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global',

 25%|██▌       | 10/40 [01:56<02:31,  5.04s/it]

❌ 분석 실패 (대외 SI/SM 사업 프로젝트 관리자(PM) 및 개발자 모집): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 34.168909404s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'glob

...

❌ 분석 실패 (신입 개발자 채용 트렌드 총정리.zip): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 8.402916681s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': '

 98%|█████████▊| 39/40 [04:39<00:05,  5.31s/it]

❌ 분석 실패 (Perception Tracking Engineer): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 50.911158448s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'm

100%|██████████| 40/40 [04:39<00:00,  6.99s/it]

❌ 분석 실패 (AI Fusion Engineer): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 50.622315075s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'loca